In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import time
import logging
LOGGER = logging.getLogger(__name__)


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- ultralytics_benchmarks_config ---

# --- ultralytics_benchmarks_dataframe ---
FIX_ULTRALYTICS_BENCHMARKS_DATAFRAME_KEY = "mAP"
FIX_ULTRALYTICS_BENCHMARKS_DATAFRAME_Y = [["ONNX","✅",5.2,0.92,12.3,81.0]]

# --- ultralytics_benchmarks_metrics ---
FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA = "coco128.yaml"
FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ = 640
FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY = "mAP"
FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL = SimpleNamespace(model_name="yolov8n")
FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0 = 0.0
FIX_ULTRALYTICS_BENCHMARKS_METRICS_VERBOSE = False
FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y = [["ONNX","✅",5.2,0.92,12.3,81.0]]

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_ultralytics_benchmarks_config():
    pd.options.display.max_columns = 10
    pd.options.display.width = 120
    return None

def before_ultralytics_benchmarks_dataframe(key, y):
    df = pd.DataFrame(y, columns=["Format", "Status❔", "Size (MB)", key, "Inference time (ms/im)", "FPS"])
    return df

def before_ultralytics_benchmarks_metrics(data, imgsz, key, model, t0, verbose, y):
    df = pd.DataFrame(y, columns=["Format", "Status\u2754", "Size (MB)", key, "Inference time (ms/im)", "FPS"])

    name = model.model_name
    dt = time.time() - t0
    legend = "Benchmarks legend:  - \u2705 Success  - \u274e Export passed but validation failed  - \u274c\ufe0f Export failed"
    s = f"\nBenchmarks complete for {name} on {data} at imgsz={imgsz} ({dt:.2f}s) {legend} {df.fillna('-')}\n"
    LOGGER.info(s)
    with open("benchmarks.log", "a", errors="ignore", encoding="utf-8") as f:
        f.write(s)

    if verbose and isinstance(verbose, float):
        metrics = df[key].array
        floor = verbose
        assert all(x > floor for x in metrics if pd.notna(x)), f"Benchmark failure: metric(s) < floor {floor}"

    return df
    return s

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_ultralytics_benchmarks_config():

    pl.Config.set_tbl_cols(10)
    pl.Config.set_tbl_width_chars(120)
    return None

def gen_ultralytics_benchmarks_dataframe(key, y):

    df = pl.DataFrame(y, schema=["Format", "Status❔", "Size (MB)", key, "Inference time (ms/im)", "FPS"])
    return df

def gen_ultralytics_benchmarks_metrics(data, imgsz, key, model, t0, verbose, y):
    import math
    import time


    df = pl.DataFrame(y, schema=["Format", "Status❔", "Size (MB)", key, "Inference time (ms/im)", "FPS"])

    name = model.model_name
    dt = time.time() - t0
    legend = "Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed"
    s = f"\nBenchmarks complete for {name} on {data} at imgsz={imgsz} ({dt:.2f}s)\n{legend}\n{df.fill_null('-').fill_nan('-')}\n"
    LOGGER.info(s)
    with open("benchmarks.log", "a", errors="ignore", encoding="utf-8") as f:
        f.write(s)

    if verbose and isinstance(verbose, float):
        metrics = df[key].to_list()
        floor = verbose
        assert all(x > floor for x in metrics if x is not None and not (isinstance(x, float) and math.isnan(x))), f"Benchmark failure: metric(s) < floor {floor}"

    return df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: ultralytics_benchmarks_metrics ===

# L1 smoke – generated
try:
    _r = gen_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, FIX_ULTRALYTICS_BENCHMARKS_METRICS_VERBOSE, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    print("✅ L1 smoke gen_ultralytics_benchmarks_metrics: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_ultralytics_benchmarks_metrics: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, FIX_ULTRALYTICS_BENCHMARKS_METRICS_VERBOSE, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    print("✅ L1 smoke before_ultralytics_benchmarks_metrics: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_ultralytics_benchmarks_metrics: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, FIX_ULTRALYTICS_BENCHMARKS_METRICS_VERBOSE, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    _rg = gen_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, FIX_ULTRALYTICS_BENCHMARKS_METRICS_VERBOSE, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    compare(_rb, _rg, "ultralytics_benchmarks_metrics")
except Exception as _e:
    print(f"❌ L2 equivalence ultralytics_benchmarks_metrics: setup error — {type(_e).__name__}: {_e}")

# L3 — verbose floor passes when all non-null metrics are above floor.
try:
    _rb = before_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, 0.5, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    _rg = gen_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, 0.5, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    compare(_rb, _rg, "L3 ultralytics_benchmarks_metrics verbose pass")
except Exception as _e:
    print(f"❌ L3 ultralytics_benchmarks_metrics verbose pass: {type(_e).__name__}: {_e}")

# L3 — verbose floor failure should raise AssertionError on both sides.
try:
    before_err = gen_err = None
    try:
        before_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, 0.95, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    except Exception as e:
        before_err = type(e)
    try:
        gen_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, 0.95, FIX_ULTRALYTICS_BENCHMARKS_METRICS_Y)
    except Exception as e:
        gen_err = type(e)
    assert before_err is AssertionError and gen_err is AssertionError
    print("✅ L3 ultralytics_benchmarks_metrics verbose fail: MATCH")
except Exception as _e:
    print(f"❌ L3 ultralytics_benchmarks_metrics verbose fail: {type(_e).__name__}: {_e}")

# L3 — null metrics are ignored by the floor check.
try:
    y = [["ONNX", "✅", 5.2, None, 12.3, 81.0], ["TorchScript", "✅", 4.8, 0.91, 10.1, 99.0]]
    _rb = before_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, 0.5, y)
    _rg = gen_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, 0.5, y)
    compare(_rb, _rg, "L3 ultralytics_benchmarks_metrics null ignored")
except Exception as _e:
    print(f"❌ L3 ultralytics_benchmarks_metrics null ignored: {type(_e).__name__}: {_e}")

# L3 — empty benchmark rows should not crash.
try:
    _rb = before_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, False, [])
    _rg = gen_ultralytics_benchmarks_metrics(FIX_ULTRALYTICS_BENCHMARKS_METRICS_DATA, FIX_ULTRALYTICS_BENCHMARKS_METRICS_IMGSZ, FIX_ULTRALYTICS_BENCHMARKS_METRICS_KEY, FIX_ULTRALYTICS_BENCHMARKS_METRICS_MODEL, FIX_ULTRALYTICS_BENCHMARKS_METRICS_T0, False, [])
    compare(_rb, _rg, "L3 ultralytics_benchmarks_metrics empty rows")
except Exception as _e:
    print(f"❌ L3 ultralytics_benchmarks_metrics empty rows: {type(_e).__name__}: {_e}")
